In [2]:
# %% [markdown]
# # 04c — Sparse MLP v2 (Retrain-Only, Fixed Init)
#
# ═══════════════════════════════════════════════════════════════════════
# WHAT THIS NOTEBOOK IS AND WHY IT EXISTS
# ═══════════════════════════════════════════════════════════════════════
#
# BUG CONFIRMED — masked-weight reinitialisation was a silent no-op,
# identical pattern to Sparse KAN's MaskedKANLinear bug:
#   `self.weight.data[q, active].uniform_(-bound, bound)` used a boolean
#   mask for advanced indexing, which returns a COPY -- the in-place
#   .uniform_() wrote to that discarded copy and never touched the real
#   tensor. Every active edge silently kept nn.Linear's own default init
#   (scaled by full in_features=1699/574), not the correct per-node
#   fan-in scale. Measured directly on the live model: mean ratio of
#   actual-max-weight to theoretical-fan-in-bound was ~0.058 (should be
#   ~1.0), matching Sparse KAN's confirmed measurement almost exactly.
#   FIXED by building a fresh row tensor and assigning it back via
#   __setitem__, which writes through correctly.
#
# DOWNSTREAM CONSEQUENCE, directly measured (not just inferred): with
# weights ~17x too small, layer-0 pre-SiLU output std was ~0.03 median,
# and SiLU(x) ~= 0.5x + 0.25x^2 near x=0 -- at that scale the quadratic
# (nonlinear) term is roughly 1-2% of the linear term. Layer 1 compounds
# this further (median ratio measured at ~0.001). In practice, Sparse
# MLP was operating close to a LINEAR model, not a genuine nonlinear one
# -- consistent with its original results tracking Ridge closely.
#
# NOTE: Sparse MLP has NO BatchNorm anywhere in its architecture (see
# sparse_mlp.py's forward(): layer0 -> SiLU -> layer1 -> SiLU -> layer2,
# nothing else). So unlike Sparse KAN, there is no second, compounding
# eps-related bug here -- the init fix alone is the complete fix for
# this model. No eps decision applies.
#
# ═══════════════════════════════════════════════════════════════════════
# WHY RETRAIN-ONLY, NOT A FRESH OPTUNA SEARCH
# ═══════════════════════════════════════════════════════════════════════
#
# Reuses the best hyperparameters already stored in the ORIGINAL Sparse
# MLP run's Optuna .db files (both no_L1 and with_L1 phases, per config,
# per seed) rather than re-searching. Same rationale as Sparse KAN v2:
# faster, and it keeps the methodological claim clean -- "hyperparameters
# were selected under the original (pre-fix) configuration and held
# fixed for the corrected retrain; no additional search budget was spent."
#
# ═══════════════════════════════════════════════════════════════════════
# RESULTS ISOLATION -- CRITICAL
# ═══════════════════════════════════════════════════════════════════════
#
# Writes to a COMPLETELY SEPARATE results directory (.../sparse_mlp_v2/)
# from the original (.../sparse_mlp/). The original checkpoints,
# predictions, and Optuna .db files are NEVER touched or overwritten --
# kept deliberately as the "before" side of a controlled ablation
# (broken-init vs fixed-init, identical hyperparameters). This notebook
# only ever READS from the original results directory's optuna/
# subfolder, and only ever WRITES to the new v2 directory.
#
# Estimated runtime: ~10 minutes for all 48 configs x 3 seeds
# (retrain only, no search -- Sparse MLP trains much faster than the KAN).

# %%
# ── COLAB SETUP ──
!pip install -q optuna

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)
from sparse_mlp import SparseMLP, sparse_mlp_weight_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("=" * 90)
print("SANITY CHECK: confirming the init fix is present in the live sparse_mlp.py")
print("=" * 90)

_fake_tax = pd.DataFrame({
    "column":        ["f1", "f2", "f3", "f4", "f5"],
    "subtheme_id":   ["01_01"]*5,
    "subtheme_name": ["SubA"]*5,
    "theme_id":      ["01"]*5,
    "theme_name":    ["ThemeX"]*5,
})
_fcols = ["f1", "f2", "f3", "f4", "f5"]
N_DUMMY = len(_fcols)

_m = SparseMLP.from_taxonomy(_fake_tax, _fcols)

# ── Check: init fix present. Only one branch here (weight) -- no
# spline_scaler, no BatchNorm, no eps/affine to check at all, since
# SparseMLP's forward() is just layer0 -> SiLU -> layer1 -> SiLU -> layer2. ──
_active = _m.layer0.mask[0].bool()
_fan_in = int(_active.sum().item())
assert _fan_in == 5, f"fixture built wrong: fan_in={_fan_in}, expected 5"
_bound = 1.0 / _fan_in**0.5
# E[max of n uniform draws] = bound * n/(n+1) = 0.833 * bound when fixed.
# Broken: weight stays at nn.Linear's default scale (full in_features),
# so ratio would be much lower than 0.83.
_ratio = _m.layer0.weight.data[0][_active].abs().max().item() / _bound
assert _ratio > 0.5, (
    f"weight ratio={_ratio:.3f} -- expected ~0.83 (fixed) vs much lower "
    f"(broken). sparse_mlp.py on Drive is STALE. STOP. Re-upload."
)
print(f"  ✓ init fix confirmed: ratio = {_ratio:.3f} "
      f"(fan_in={_fan_in}, expect ~0.83)")

# ── Masking check, and a forward pass to confirm the model runs end-to-end ──
assert _m.verify_masking(), "masking broken at construction. STOP."
_m.eval()
assert _m(torch.randn(16, N_DUMMY)).shape == (16, 1)
print("  ✓ forward pass OK (batch=16)")

del _fake_tax, _fcols, _m, _active, _fan_in, _bound, _ratio, N_DUMMY

print("\n" + "=" * 90)
print("SANITY CHECK PASSED -- safe to proceed with retrain-only sweep")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

# ── READ from the ORIGINAL results (Optuna .db files only) ──
ORIGINAL_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_mlp")

# ── WRITE to a completely separate v2 directory. NEVER touches the original. ──
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_mlp_v2")

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

SEEDS = [42, 123, 456]

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════
# LOAD TAXONOMIES
# ═══════════════════════════════════════════════════════════════════════════

print("Loading taxonomies...")
taxonomy_dfs = {}
for ds in DATASETS:
    df = load_theme_assignment(ds, THEMES_DIR)
    taxonomy_dfs[ds] = df
    print(f"  {ds}: {len(df)} features, "
          f"{df['subtheme_id'].nunique()} subthemes, "
          f"{df['theme_id'].nunique()} themes")

device = get_device()


# ═══════════════════════════════════════════════════════════════════════════
# RETRAIN-ONLY: load stored best params, skip Optuna search entirely
# ═══════════════════════════════════════════════════════════════════════════

def load_best_params_from_original(model_name, target_type, split_name, seed):
    """
    Reads the ORIGINAL run's Optuna studies (no_L1 and with_L1 phases) for
    this exact config, picks whichever phase originally won, and returns
    its best_params -- WITHOUT running any new trials. Raises loudly if
    the original study is missing, rather than silently falling back to a
    fresh search (which would break the "no additional tuning" guarantee).
    """
    study_dir = ORIGINAL_RESULTS_DIR / f"seed_{seed}" / "optuna"

    path_no_l1 = study_dir / f"{model_name}_{target_type}_{split_name}_no_L1_seed{seed}.db"
    path_l1    = study_dir / f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}.db"

    if not path_no_l1.exists() or not path_l1.exists():
        raise FileNotFoundError(
            f"Original Optuna studies not found for {model_name}/{target_type}/"
            f"{split_name}/seed{seed} at {study_dir} -- cannot retrain-only "
            f"without the original hyperparameter search results."
        )

    study_no_l1 = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_no_L1_seed{seed}",
        storage=f"sqlite:///{path_no_l1}",
    )
    study_l1 = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}",
        storage=f"sqlite:///{path_l1}",
    )

    if study_no_l1.best_value >= study_l1.best_value:
        return study_no_l1.best_params, False, study_no_l1.best_value
    else:
        return study_l1.best_params, True, study_l1.best_value


def run_single_retrain(split_name, dataset, target_type, device,
                       seed, seed_results_dir):
    model_name = f"sparse_mlp_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None

    best_params, use_l1, original_val = load_best_params_from_original(
        model_name, target_type, split_name, seed
    )
    print(f"  Reusing stored params (original val={original_val:.4f}, "
          f"used_l1={use_l1}): {best_params}")

    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = SparseMLP.from_taxonomy(taxonomy_df, feature_cols)

    train_kwargs = {
        "lr":           best_params["lr"],
        "weight_decay": best_params["weight_decay"],
        "pos_weight":   pos_weight if target_type == "binary" else None,
        "n_epochs":     300,
        "patience":     20,
        "verbose":      True,
        "log_every":    20,
    }
    if use_l1:
        train_kwargs["reg_fn"]     = sparse_mlp_weight_l1
        train_kwargs["reg_weight"] = best_params["reg_weight"]
    if target_type == "continuous":
        train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        **train_kwargs,
    )

    assert model.verify_masking(), (
        "Masked weights non-zero after final training completed -- "
        "structural sparsity has been broken somewhere."
    )

    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=(
                {**best_params, "used_l1": use_l1, "seed": seed,
                 "huber_delta": huber_delta,
                 "reused_from_original": True,
                 "original_val_metric": original_val}
                if part == "test" else None
            ),
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        all_metrics[part] = metrics

    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "used_l1": use_l1, "seed": seed,
                         "huber_delta": huber_delta,
                         "reused_from_original": True},
        model_config={
            "type":              "SparseMLP_Masked_v2",
            "dataset":           dataset,
            "target_type":       target_type,
            "n_features":        data["n_features"],
            "n_subthemes":       model.n_subthemes,
            "n_themes":          model.n_themes,
            "activation":        "SiLU",
            "active_edges":      model.count_active_edges(),
            "active_parameters": model.count_active_parameters(),
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    if target_type == "binary":
        print(f"\n  Results (seed={seed}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       {all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R2: {all_metrics['train']['r2']:.4f}")
        print(f"    Val R2:   {all_metrics['val']['r2']:.4f}")
        print(f"    Test R2:  {all_metrics['test']['r2']:.4f}")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")

    return {
        "best_params": best_params,
        "used_l1":     use_l1,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Run Retrain-Only Sweep (48 configs x 3 seeds, no Optuna search)

# %%
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
total_runs = len(SEEDS) * configs

print("=" * 70)
print(f"  SPARSE MLP v2 (RETRAIN-ONLY): {total_runs} runs")
print(f"  Reusing hyperparameters from: {ORIGINAL_RESULTS_DIR}")
print(f"  Writing NEW results to:       {RESULTS_DIR}")
print(f"  Original results directory is NEVER written to.")
print(f"  Init fix applied (was silently no-op). No eps -- SparseMLP has no BatchNorm.")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_retrain(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        "used_l1": exp["used_l1"],
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = {**exp["best_params"], "used_l1": exp["used_l1"]}
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.1f}min elapsed, ~{remaining_est/60:.1f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes")
print(f"{'='*70}")

# %%
if all_results:
    results_df = pd.DataFrame(all_results)
    raw_path = RESULTS_DIR / "all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"Raw results saved to {raw_path}")

    binary_df = results_df[results_df["target"] == "binary"]
    print("\nBinary Test AUC (mean ± std across seeds):")
    if "test_auc" in binary_df.columns:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(["mean", "std"])
        print(agg.to_string())

    cont_df = results_df[results_df["target"] == "continuous"]
    print("\nContinuous Test R² (mean ± std across seeds):")
    if "test_r2" in cont_df.columns:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(["mean", "std"])
        print(agg.to_string())


# %% [markdown]
# ## Backtests (Seed-Averaged Signal)
#
# Reads from THIS notebook's v2 results directory only -- does not touch
# or read from the original sparse_mlp/ results in any way.

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                               seeds, results_dir):
    signals = []
    returns = None

    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            results_dir=seed_dir,
        )
        preds = loaded["predictions"]

        if returns is None:
            returns = preds["daily_return"].values

        if target_type == "binary":
            signals.append(preds["y_prob"].values)
        else:
            signals.append(preds["y_pred"].values)

    avg_signal = np.mean(np.stack(signals, axis=0), axis=0)
    return returns, avg_signal


def _make_json_safe(obj):
    if isinstance(obj, dict): return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame): return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj


# %%
print("\n" + "=" * 70)
print("  SPARSE MLP v2 — BACKTESTS (seed-averaged signal)")
print("  Signal = mean prediction across seeds 42, 123, 456")
print("  Reading from v2 results only -- original sparse_mlp/ untouched")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)

backtest_rows    = []
backtest_records = {}

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"sparse_mlp_{dataset}"

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)

        bt_binary = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="above",
            model_name=f"{model_name}_v2 (binary)",
            split_name=split_name,
        )

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)

        bt_continuous = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="below",
            model_name=f"{model_name}_v2 (continuous)",
            split_name=split_name,
        )

        key = f"{dataset}/{split_name}"
        backtest_records[key] = {"binary": bt_binary, "continuous": bt_continuous}

        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset, "split": split_name,
                    "target_type": target_type, "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"], "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"],
                    "max_drawdown": bt_result["max_drawdown"],
                    "cumulative_return": bt_result["cumulative_return"],
                    "avg_exposure": bt_result["avg_exposure"],
                    "annual_turnover": bt_result["annual_turnover"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                    "buy_hold_sortino": bt_result["buy_hold_sortino"],
                    "buy_hold_cumulative": bt_result["buy_hold_cumulative"],
                })

backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results saved to {backtest_json_path}")


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (v2 -- on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("sparse_mlp_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")

for fname in ["all_seeds_raw.csv"]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        print(f"\n  {fname}: ✓")
    else:
        print(f"\n  {fname}: (not yet created)")

print(f"\n  backtests/: {'✓' if (RESULTS_DIR / 'backtests' / 'backtest_summary.csv').exists() else '(not yet created)'}")

print(f"\n  Original results directory (unchanged, kept as ablation baseline):")
print(f"    {ORIGINAL_RESULTS_DIR}")


# %% [markdown]
# ## Disconnect Runtime

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
SANITY CHECK: confirming the init fix is present in the live sparse_mlp.py
  ✓ init fix confirmed: ratio = 0.878 (fan_in=5, expect ~0.83)
  ✓ All masked parameters are exactly zero
  ✓ forward pass OK (batch=16)

SANITY CHECK PASSED -- safe to proceed with retrain-only sweep
Loading taxonomies...
  agg_full_moments: 1699 features, 331 subthemes, 13 themes
  agg_means: 574 features, 128 subthemes, 13 themes
Device: Tesla T4 (CUDA)
  SPARSE MLP v2 (RETRAIN-ONLY): 48 runs
  Reusing hyperparameters from: /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_mlp
  Writing NEW results to:       /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_mlp_v2
  Original results directory is NEVER written to.
  Init fix applied (was silently no-op). No eps -- SparseMLP has no BatchNorm.


══════════════════════════════════════════════════════════════════════
  SEED 42 — saving to /content